# LoRA Fine-Tune MegaDescriptor (Colab + Drive)

Fine-tune **MegaDescriptor-L-384** with **LoRA + ArcFace** on Amvrakikos + Reunion, then evaluate opposite-side re-ID on held-out **Zakynthos**.

## Before you run
1. Open this notebook in **Google Colab**.
2. Set **Runtime → Change runtime type → GPU** (T4 or better).
3. Put datasets under Drive:
   ```
   MyDrive/SeaTurtle/
     AmvrakikosTurtles/
     ReunionTurtles/
     ZakynthosTurtles/
   ```
4. Run cells top to bottom. Checkpoints and features are saved back to Drive.

In [ ]:
import os
import sys
import subprocess

def assert_colab_gpu():
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError(
            'This notebook is Colab-only. Open it in Google Colab with a GPU runtime.'
        ) from exc
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            'CUDA GPU is required. In Colab choose Runtime > Change runtime type > GPU.'
        )
    print(f'Colab GPU: {torch.cuda.get_device_name(0)}')
    return torch

def find_repo_root():
    path = os.path.abspath(os.getcwd())
    for _ in range(6):
        if os.path.isdir(os.path.join(path, 'sides_matching')):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            break
        path = parent
    for candidate in ['/content/sides-matching', '/content']:
        if os.path.isdir(os.path.join(candidate, 'sides_matching')):
            return candidate
    return None

def ensure_repo_root():
    root = find_repo_root()
    if root is not None:
        return root
    clone_dir = '/content/sides-matching'
    print('Cloning sides-matching repo...')
    subprocess.check_call([
        'git', 'clone', '--depth', '1',
        'https://github.com/sadda/sides-matching.git', clone_dir,
    ])
    return clone_dir

def pip_install(*packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *packages]
    print('>', ' '.join(cmd))
    subprocess.check_call(cmd)

torch = assert_colab_gpu()
repo_root = ensure_repo_root()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

pip_install('wildlife-datasets', 'timm', 'scikit-image', 'peft')
pip_install('git+https://github.com/WildlifeDatasets/wildlife-tools@main')

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = '/content/drive/MyDrive/SeaTurtle'
DATASET_DIRS = ('AmvrakikosTurtles', 'ReunionTurtles', 'ZakynthosTurtles')

def has_datasets(path):
    return all(os.path.isdir(os.path.join(path, name)) for name in DATASET_DIRS)

if not has_datasets(DRIVE_DATA):
    raise FileNotFoundError(
        f'Datasets not found at {DRIVE_DATA}. '
        'Put AmvrakikosTurtles, ReunionTurtles, ZakynthosTurtles in Drive > SeaTurtle.'
    )

device = torch.device('cuda')
root_features = os.path.join(DRIVE_DATA, 'features')
checkpoint_dir = os.path.join(DRIVE_DATA, 'checkpoints', 'lora_megadescriptor')
os.makedirs(root_features, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

print(f'Repo: {repo_root}')
print(f'Data: {DRIVE_DATA} OK')
print(f'Features: {root_features}')
print(f'Checkpoints: {checkpoint_dir}')

In [ ]:
import pandas as pd
from sides_matching import amvrakikos, reunion_green, reunion_hawksbill, zakynthos
from sides_matching.train_lora import (
    TrainConfig,
    IdentityMapper,
    merge_train_dataframes,
    split_identities,
    get_train_transform,
    get_eval_transform,
    wildlife_dataset_from_df,
    build_concat_dataloader,
)

config = TrainConfig()
train_transform = get_train_transform(config.img_size)
val_transform = get_eval_transform(flip=False, img_size=config.img_size)

root_data = DRIVE_DATA
train_sources = [
    ('Amvrakikos', os.path.join(root_data, 'AmvrakikosTurtles'), amvrakikos),
    ('ReunionGreen', os.path.join(root_data, 'ReunionTurtles'), reunion_green),
    ('ReunionHawksbill', os.path.join(root_data, 'ReunionTurtles'), reunion_hawksbill),
]

train_frames = []
for name, root, dataset_fn in train_sources:
    dataset = dataset_fn(root, transform=None)
    train_frames.append((name, dataset.df.copy()))
    print(f'{name}: {len(dataset.df)} images, {dataset.df["identity"].nunique()} identities')

merged_df = merge_train_dataframes(train_frames)
train_df, val_df = split_identities(
    merged_df,
    val_fraction=config.val_identity_fraction,
    seed=config.seed,
)
identity_mapper = IdentityMapper.from_identities(merged_df['global_identity'])

print(f'Train images: {len(train_df)} | Val images: {len(val_df)} | Classes: {identity_mapper.num_classes}')

train_datasets = []
train_label_parts = []
val_datasets = []
val_label_parts = []

for name, root, dataset_fn in train_sources:
    source_train = train_df[train_df['source_dataset'] == name].reset_index(drop=True)
    source_val = val_df[val_df['source_dataset'] == name].reset_index(drop=True)
    if len(source_train):
        train_datasets.append(wildlife_dataset_from_df(root, source_train, dataset_fn, train_transform))
        train_label_parts.append(identity_mapper.encode_series(source_train['global_identity']))
    if len(source_val):
        val_datasets.append(wildlife_dataset_from_df(root, source_val, dataset_fn, val_transform))
        val_label_parts.append(identity_mapper.encode_series(source_val['global_identity']))

train_loader = build_concat_dataloader(train_datasets, train_label_parts, config.batch_size, shuffle=True)
val_loader = build_concat_dataloader(val_datasets, val_label_parts, config.batch_size, shuffle=False)

zakynthos_root = os.path.join(root_data, 'ZakynthosTurtles')
zakynthos_dataset = zakynthos(zakynthos_root, transform=None)
print(f'Zakynthos (test only): {len(zakynthos_dataset.df)} images, {zakynthos_dataset.df["identity"].nunique()} identities')

In [ ]:
from sides_matching.train_lora import build_training_model, configure_optimizer

model = build_training_model(identity_mapper.num_classes, config, device)
optimizer = configure_optimizer(model, config)
model.backbone.print_trainable_parameters()

In [ ]:
from sides_matching.train_lora import train_epoch, validate_epoch, embedding_recall_at_1, save_checkpoint

best_recall = -1.0
patience_counter = 0
history = []

for epoch in range(1, config.epochs + 1):
    try:
        train_loss = train_epoch(model, train_loader, optimizer, device)
    except RuntimeError as exc:
        if 'out of memory' in str(exc).lower() and config.batch_size > 8:
            config.batch_size = 8
            train_loader = build_concat_dataloader(train_datasets, train_label_parts, config.batch_size, shuffle=True)
            val_loader = build_concat_dataloader(val_datasets, val_label_parts, config.batch_size, shuffle=False)
            print('CUDA OOM: reduced batch_size to 8, retrying epoch...')
            train_loss = train_epoch(model, train_loader, optimizer, device)
        else:
            raise

    val_loss, val_recall = validate_epoch(model, val_loader, device)
    embed_recall = embedding_recall_at_1(model, val_loader, device)
    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_recall': val_recall,
        'embed_recall': embed_recall,
    })
    print(
        f'Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | '
        f'val_recall@1={val_recall:.4f} | embed_recall@1={embed_recall:.4f}'
    )

    if embed_recall > best_recall:
        best_recall = embed_recall
        patience_counter = 0
        save_checkpoint(
            model,
            identity_mapper,
            config,
            checkpoint_dir,
            metrics={'embed_recall_at_1': embed_recall, 'epoch': epoch},
        )
        print(f'Saved best checkpoint to {checkpoint_dir}')
    else:
        patience_counter += 1
        if patience_counter >= config.early_stop_patience:
            print('Early stopping triggered.')
            break

history_df = pd.DataFrame(history)
display(history_df.tail())

In [ ]:
from sides_matching.train_lora import build_inference_model, extract_features, save_feature_pickle, get_eval_transform
from wildlife_tools.features import DeepFeatures
from sides_matching import get_features, get_transform
import timm

adapter_dir = os.path.join(checkpoint_dir, 'adapter')
if not os.path.isdir(adapter_dir):
    raise FileNotFoundError(f'Missing adapter at {adapter_dir}. Run training first.')

lora_model = build_inference_model(config, adapter_dir, device)
grayscale = False

for flip in [True, False]:
    transform = get_eval_transform(flip=flip, img_size=config.img_size)
    eval_dataset = zakynthos(zakynthos_root, transform=transform)
    features = extract_features(lora_model, eval_dataset, device, batch_size=config.batch_size)
    file_name = os.path.join(
        root_features,
        f'MegaDescriptorLoRA_Zakynthos_flip={flip}_grayscale={grayscale}.pickle',
    )
    save_feature_pickle(features, file_name)
    print(f'Saved {file_name} shape={features.shape}')

baseline_model = timm.create_model(config.model_name, num_classes=0, pretrained=True).to(device)
baseline_model.eval()
baseline_extractor = DeepFeatures(baseline_model, batch_size=config.batch_size, device=device)

for flip in [True, False]:
    transform = get_transform(flip=flip, grayscale=grayscale, img_size=config.img_size, normalize=True)
    eval_dataset = zakynthos(zakynthos_root, transform=transform)
    file_name = os.path.join(
        root_features,
        f'MegaDescriptor_Zakynthos_flip={flip}_grayscale={grayscale}.pickle',
    )
    if not os.path.exists(file_name):
        get_features(file_name, eval_dataset, baseline_extractor)
        print(f'Extracted baseline {file_name}')
    else:
        print(f'Using existing baseline {file_name}')

In [ ]:
from sides_matching.train_lora import evaluate_zakynthos_predictions

mods = ['full', 'same orientation', 'different orientation', 'same year', 'different year', 'different both']
results = []

for flip in [True, False]:
    for method_prefix in ['MegaDescriptor', 'MegaDescriptorLoRA']:
        part = evaluate_zakynthos_predictions(
            zakynthos_dataset.df,
            root_features,
            method_prefix=method_prefix,
            flip=flip,
            grayscale=False,
            mods=mods,
        )
        results.append(part)

results_df = pd.concat(results, ignore_index=True)
pivot = results_df.pivot_table(
    index=['method', 'flip', 'mod'],
    values=['top1', 'top5'],
    aggfunc='first',
)
display(pivot)

results_csv = os.path.join(DRIVE_DATA, 'checkpoints', 'lora_megadescriptor', 'zakynthos_eval.csv')
results_df.to_csv(results_csv, index=False)
print(f'Saved evaluation table to {results_csv}')